# Assignment 2

In this assigment, we will work with the *Adult* data set. Please download the data from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/2/adult). Extract the data files into the subdirectory: `../05_src/data/adult/` (relative to `./05_src/`).

# Load the data

Assuming that the files `adult.data` and `adult.test` are in `../05_src/data/adult/`, then you can use the code below to load them.

In [240]:
import pandas as pd
columns = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status',
    'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week',
    'native-country', 'income'
]
adult_dt = (pd.read_csv("../../05_src/data/adult/adult.data", header = None, names = columns)
              .assign(income = lambda x: (x.income.str.strip() == '>50K')*1))
adult_dt.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       32561 non-null  object
 2   fnlwgt          32561 non-null  int64 
 3   education       32561 non-null  object
 4   education-num   32561 non-null  int64 
 5   marital-status  32561 non-null  object
 6   occupation      32561 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital-gain    32561 non-null  int64 
 11  capital-loss    32561 non-null  int64 
 12  hours-per-week  32561 non-null  int64 
 13  native-country  32561 non-null  object
 14  income          32561 non-null  int32 
dtypes: int32(1), int64(6), object(8)
memory usage: 3.6+ MB


# Get X and Y

Create the features data frame and target data:

+ Create a dataframe `X` that holds the features (all columns that are not `income`).
+ Create a dataframe `Y` that holds the target data (`income`).
+ From `X` and `Y`, obtain the training and testing data sets:

    - Use a train-test split of 70-30%. 
    - Set the random state of the splitting function to 42.

In [ ]:
from sklearn.model_selection import train_test_split
X = adult_dt[['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status',
    'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week',
    'native-country']]
Y = adult_dt['income']

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=42)

## Random States

Please comment: 

+ What is the [random state](https://scikit-learn.org/stable/glossary.html#term-random_state) of the [splitting function](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)? 
+ Why is it [useful](https://en.wikipedia.org/wiki/Reproducibility)?

*(Comment here.)*\
Random state is passing an integer for a reproducible split output. It is setting a seed that fixes random number generator acting within the split function by pointing to a specific sequence of generated random numbers. By calling a particular state, a random split can be replicated.

# Preprocessing

Create a [Column Transformer](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html) that treats the features as follows:

- Numerical variables

    * Apply [KNN-based imputation for completing missing values](https://scikit-learn.org/stable/modules/generated/sklearn.impute.KNNImputer.html):
        
        + Consider the 7 nearest neighbours.
        + Weight each neighbour by the inverse of its distance, causing closer neigbours to have more influence than more distant ones.
    * [Scale features using statistics that are robust to outliers](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.RobustScaler.html#sklearn.preprocessing.RobustScaler).

- Categorical variables: 
    
    * Apply a [simple imputation strategy](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html#sklearn.impute.SimpleImputer):

        + Use the most frequent value to complete missing values, also called the *mode*.

    * Apply [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html):
        
        + Handle unknown labels if they exist.
        + Drop one column for binary variables.
    
    
The column transformer should look like this:

![](./images/assignment_2__column_transformer.png)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import KNNImputer
from sklearn.pipeline import Pipeline

num_transforms = Pipeline([
                ('num_imputer', KNNImputer(n_neighbors=7, weights='distance')),
                ('num_robustscaler', RobustScaler()), 
            ])
cat_transforms = Pipeline([
                ('cat_onehot', SimpleImputer(strategy='most_frequent')),
                ('cat_transforms', OneHotEncoder(handle_unknown='infrequent_if_exist')),   
            ])

transformer = ColumnTransformer(
    transformers=
    [
        ('num_transforms', num_transforms, ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']),
        ('cat_transforms', cat_transforms, ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']),
    ],
    remainder = 'drop'
)
transformer

ColumnTransformer(transformers=[('num_transforms',
                                 Pipeline(steps=[('num_imputer',
                                                  KNNImputer(n_neighbors=7,
                                                             weights='distance')),
                                                 ('num_robustscaler',
                                                  RobustScaler())]),
                                 ['age', 'fnlwgt', 'education-num',
                                  'capital-gain', 'capital-loss',
                                  'hours-per-week']),
                                ('cat_transforms',
                                 Pipeline(steps=[('cat_onehot',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('cat_transforms',
                                                  OneHotEncoder(handle_unknown='infrequent_if_exist'))]),
                                 ['workclass', 'education', 'marital-status',
                                  'occupation', 'relationship', 'race', 'sex',
                                  'native-country'])])

## Model Pipeline

Create a [model pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): 

+ Add a step labelled `preprocessing` and assign the Column Transformer from the previous section.
+ Add a step labelled `classifier` and assign a [`RandomForestClassifier()`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) to it.

The pipeline looks like this:

![](./images/assignment_2__pipeline.png)

In [220]:
from sklearn.ensemble import RandomForestClassifier 

pipe = Pipeline(
    [
        ('preprocessing', transformer), 
        ('randomforest', RandomForestClassifier())
    ]
)

pipe

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num_transforms',
                                                  Pipeline(steps=[('num_imputer',
                                                                   KNNImputer(n_neighbors=7,
                                                                              weights='distance')),
                                                                  ('num_robustscaler',
                                                                   RobustScaler())]),
                                                  ['age', 'fnlwgt',
                                                   'education-num',
                                                   'capital-gain',
                                                   'capital-loss',
                                                   'hours-per-week']),
                                                 ('cat_transforms',
                                                  Pipeline(steps=[('cat_onehot',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('cat_transforms',
                                                                   OneHotEncoder(handle_unknown='infrequent_if_exist'))]),
                                                  ['workclass', 'education',
                                                   'marital-status',
                                                   'occupation', 'relationship',
                                                   'race', 'sex',
                                                   'native-country'])])),
                ('randomforest', RandomForestClassifier())])

# Cross-Validation

Evaluate the model pipeline using [`cross_validate()`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_validate.html):

+ Measure the following [preformance metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#common-cases-predefined-values): negative log loss, ROC AUC, accuracy, and balanced accuracy.
+ Report the training and validation results. 
+ Use five folds.


In [ ]:
from sklearn.metrics import accuracy_score, log_loss, balanced_accuracy_score, roc_auc_score
from sklearn.model_selection import cross_validate

scoring = ['neg_log_loss', 'roc_auc',  'accuracy', 'balanced_accuracy']
train_results = cross_validate(pipe, X_train, Y_train, cv=5, scoring = scoring, return_train_score = True, error_score='raise')


Display the fold-level results as a pandas data frame and sorted by negative log loss of the test (validation) set.

In [223]:
train_results = pd.DataFrame(train_results)
results_sorted = train_results.sort_values('test_neg_log_loss')
results_sorted

,fit_time,score_time,test_neg_log_loss,train_neg_log_loss,test_roc_auc,train_roc_auc,test_accuracy,train_accuracy,test_balanced_accuracy,train_balanced_accuracy
2,8.591343,0.171290,-0.391093,-0.081519,0.901305,1.0,0.850373,0.999945,0.769555,0.999887
4,8.396563,0.173983,-0.381697,-0.081679,0.901556,1.0,0.854980,1.000000,0.774750,1.000000
3,9.442369,0.207442,-0.363158,-0.081794,0.907237,1.0,0.858052,0.999945,0.781700,0.999964
1,9.177096,0.173041,-0.347755,-0.081209,0.902350,1.0,0.850186,0.999945,0.772658,0.999887
0,8.068307,0.175290,-0.346909,-0.081176,0.906490,1.0,0.849748,1.000000,0.775749,1.000000


Calculate the mean of each metric. 

In [224]:
results_sorted.mean()


fit_time                   8.735135
score_time                 0.180209
test_neg_log_loss         -0.366123
train_neg_log_loss        -0.081475
test_roc_auc               0.903788
train_roc_auc              1.000000
test_accuracy              0.852668
train_accuracy             0.999967
test_balanced_accuracy     0.774882
train_balanced_accuracy    0.999948
dtype: float64

In [228]:
def evaluate_model(pipe, X_train, Y_train, X_test, Y_test):
    pipe.fit(X_train, Y_train)
    Y_pred_test = pipe.predict(X_test)
    Y_proba_test = pipe.predict_proba(X_test)
    performance_metrics = {
        'log_loss': log_loss(Y_test, Y_proba_test),
        'roc_auc_test': roc_auc_score(Y_test, Y_proba_test[:, 1]),
        'accuracy_score': accuracy_score(Y_test, Y_pred_test),
        'balanced_accuracy': balanced_accuracy_score(Y_test, Y_pred_test)
    }
    return performance_metrics

results_test = evaluate_model(pipe, X_train, Y_train, X_test, Y_test)
results_test


{'log_loss': 0.3670907124695214,
 'roc_auc_test': 0.9016708432676149,
 'accuracy_score': 0.8537209540382844,
 'balanced_accuracy': 0.7748202264581439}

Calculate the same performance metrics (negative log loss, ROC AUC, accuracy, and balanced accuracy) using the testing data `X_test` and `Y_test`. Display results as a dictionary.

*Tip*: both, `roc_auc()` and `neg_log_loss()` will require prediction scores from `pipe.predict_proba()`. However, for `roc_auc()` you should only pass the last column `Y_pred_proba[:, 1]`. Use `Y_pred_proba` with `neg_log_loss()`.

# Target Recoding

In the first code chunk of this document, we loaded the data and immediately recoded the target variable `income`. Why is this [convenient](https://scikit-learn.org/stable/modules/model_evaluation.html#binary-case)?

The specific line was:

```
adult_dt = (pd.read_csv('../05_src/data/adult/adult.data', header = None, names = columns)
              .assign(income = lambda x: (x.income.str.strip() == '>50K')*1))
```

(Answer here.) \
It's convinient beacuse we are processing the target variable right away with a unique transformation that is needed for the targer variable only and can be introduced during data loading instead of including in a pipeline. The transformation checks for a condidion with the .strip method and converts the boolean result into a binary product (*1). 

## Criteria

The [rubric](./assignment_2_rubric_clean.xlsx) contains the criteria for assessment.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-2`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_2.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.

# Reference

Becker,Barry and Kohavi,Ronny. (1996). Adult. UCI Machine Learning Repository. https://doi.org/10.24432/C5XW20.